In [ ]:
import os, subprocess, sys

# ── Colab bootstrap: clone repo & chdir into it ──
if 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules:
    REPO_DIR = '/content/hetnet-traffic-forecast'
    if not os.path.isdir(REPO_DIR):
        print('Cloning repository …')
        subprocess.check_call(
            ['git', 'clone', 'https://github.com/Sreekar-2612/hetnet-traffic-forecast.git', REPO_DIR])
    os.chdir(REPO_DIR)
    # Stash for later cells so they never lose track of the repo root
    __NB_DIR__ = REPO_DIR
    print(f'Colab ready  ·  CWD = {os.getcwd()}')
else:
    print('Not Colab – skipping clone step.')


# TASTF: Tier-Aware Spatiotemporal Forecasting for HetNets

This notebook implements the **TASTF** (Tier-Aware Spatiotemporal Forecasting) model for Heterogeneous Cellular Networks (HetNets). 

### Core Contribution
TASTF uses a **Heterogeneous Graph Convolutional Network** that treats Macro, Pico, and Femto base stations as distinct node types with type-specific convolutional layers via PyTorch Geometric `HeteroConv`. This approach explicitly models the hierarchy and cross-tier influence in HetNets, which is then processed by a **Transformer Encoder** to capture long-range temporal dependencies.

---

## 1. Environment Setup & Dependencies

In [ ]:
import os, sys
from pathlib import Path

# ── Universal reference_implementation/ finder ──
# Works on Colab, VS Code, Jupyter, local terminal – no hardcoded user paths.

def _find_ref_impl():
    """Return (repo_root, ref_impl_dir) or raise FileNotFoundError."""
    seeds = []

    # 1. __NB_DIR__ set by the Colab bootstrap cell
    nb_dir = globals().get('__NB_DIR__')
    if nb_dir:
        seeds.append(Path(nb_dir))

    # 2. VS Code / IPython notebook path
    import inspect
    frame = inspect.currentframe()
    try:
        while frame:
            vsc = frame.f_globals.get('__vsc_ipynb_file__')
            if vsc:
                seeds.append(Path(vsc).resolve().parent)
                break
            frame = frame.f_back
    finally:
        del frame

    # 3. CWD and ancestors (up to 6 levels)
    cwd = Path.cwd().resolve()
    curr = cwd
    for _ in range(6):
        seeds.append(curr)
        if curr == curr.parent:
            break
        curr = curr.parent

    # 4. Common Colab layout
    seeds.append(Path('/content/hetnet-traffic-forecast'))

    # De-duplicate while preserving order
    seen, unique = set(), []
    for s in seeds:
        key = str(s)
        if key not in seen:
            seen.add(key); unique.append(s)

    for root in unique:
        ref = root / 'reference_implementation'
        if ref.is_dir():
            return root, ref

    raise FileNotFoundError(
        'reference_implementation/ not found. '
        f'CWD={cwd} | searched={[str(s) for s in unique]}  ·  '
        'On Colab, make sure Cell-1 (clone step) ran first.')

REPO, REF_DIR = _find_ref_impl()
if str(REF_DIR) not in sys.path:
    sys.path.insert(0, str(REF_DIR))
print(f'repo root  : {REPO}')
print(f'ref_impl   : {REF_DIR}')

from train import run_training
from paths import resolve_wireless_dataset_dir

DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)


In [ ]:
# ── Metrics & figures cell ──
# Reuse REPO / REF_DIR from the training cell above.
# If you ran only this section, the finder will re-discover the paths.
import sys
from pathlib import Path

if 'REF_DIR' not in dir():
    REPO, REF_DIR = _find_ref_impl()
    if str(REF_DIR) not in sys.path:
        sys.path.insert(0, str(REF_DIR))

results_path = REF_DIR / 'results.npz'
print(f'Looking for results at: {results_path}')
if not results_path.exists():
    raise FileNotFoundError(
        f'{results_path} not found – run the Training cell (Section 2) first.')
print('results.npz found ✓')


## 2. Training (canonical pipeline)

Runs the same code as 
eference_implementation/train.py: baselines, chronological split, temporal features, inverse metrics.  
**Data:** place Milan *.txt files in hetnet-traffic-forecast/Wireless Dataset/ (or set DATA_DIR below).


In [3]:
import os
import sys
from pathlib import Path

# Debugging information
print(f"Current working directory: {Path.cwd()}")
print(f"Notebook file resolve: {Path('TASTF_Implementation.ipynb').resolve()}")

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'hetnet-traffic-forecast',
    Path(r'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    Path('/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    # Try to find it relative to the notebook if it's in a subfolder or something
    Path(__file__).parent if '__file__' in locals() else Path('.'),
]

found = False
for root in candidates:
    ref = root / 'reference_implementation'
    print(f"Checking: {ref}")
    if ref.is_dir():
        print(f"Success! Found at: {ref}")
        if str(ref) not in sys.path:
            sys.path.insert(0, str(ref))
        REPO = root
        found = True
        break

if not found:
    # Aggressive search in parent directories
    curr = Path.cwd().resolve()
    for _ in range(5):
        ref = curr / 'reference_implementation'
        print(f"Checking parent: {ref}")
        if ref.is_dir():
            print(f"Success! Found at: {ref}")
            if str(ref) not in sys.path:
                sys.path.insert(0, str(ref))
            REPO = curr
            found = True
            break
        if curr == curr.parent:
            break
        curr = curr.parent

if not found:
    raise FileNotFoundError('reference_implementation/ not found. Diagnostic: ' + 
                            f'CWD={Path.cwd()} | candidates={[str(c) for c in candidates]}')

from train import run_training
from paths import resolve_wireless_dataset_dir

DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)


Current working directory: /content
Notebook file resolve: /content/TASTF_Implementation.ipynb
Checking: /content/reference_implementation
Checking: /reference_implementation
Checking: /content/hetnet-traffic-forecast/reference_implementation
Checking: c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project/reference_implementation
Checking: /mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project/reference_implementation
Checking: reference_implementation
Checking parent: /content/reference_implementation
Checking parent: /reference_implementation


FileNotFoundError: reference_implementation/ not found. Diagnostic: CWD=/content | candidates=['/content', '/', '/content/hetnet-traffic-forecast', 'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project', '/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project', '.']

## 3. Metrics & figures

Requires 
eference_implementation/results.npz from the training cell.


In [ ]:
import os
import sys
from pathlib import Path

# Debugging information
print(f"Current working directory: {Path.cwd()}")
print(f"Notebook file resolve: {Path('TASTF_Implementation.ipynb').resolve()}")

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'hetnet-traffic-forecast',
    Path(r'c:/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    Path('/mnt/c/Users/sreek/OneDrive/Desktop/ACADS-ASSIGNS/wireless/project'),
    # Try to find it relative to the notebook if it's in a subfolder or something
    Path(__file__).parent if '__file__' in locals() else Path('.'),
]

found = False
for root in candidates:
    ref = root / 'reference_implementation'
    print(f"Checking: {ref}")
    if ref.is_dir():
        print(f"Success! Found at: {ref}")
        if str(ref) not in sys.path:
            sys.path.insert(0, str(ref))
        REPO = root
        found = True
        break

if not found:
    # Aggressive search in parent directories
    curr = Path.cwd().resolve()
    for _ in range(5):
        ref = curr / 'reference_implementation'
        print(f"Checking parent: {ref}")
        if ref.is_dir():
            print(f"Success! Found at: {ref}")
            if str(ref) not in sys.path:
                sys.path.insert(0, str(ref))
            REPO = curr
            found = True
            break
        if curr == curr.parent:
            break
        curr = curr.parent

if not found:
    raise FileNotFoundError('reference_implementation/ not found. Diagnostic: ' + 
                            f'CWD={Path.cwd()} | candidates={[str(c) for c in candidates]}')

from train import run_training
from paths import resolve_wireless_dataset_dir

DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)
